In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

np.random.seed(42)
tf.random.set_seed(42)

vocab_size = 10000
max_len = 200

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

train_size = 10000
test_size = 2500

x_train_sub = x_train[:train_size]
y_train_sub = y_train[:train_size]

x_test_sub = x_test[:test_size]
y_test_sub = y_test[:test_size]

x_train_padded = pad_sequences(x_train_sub, maxlen=max_len, padding='post', truncating='post')
x_test_padded = pad_sequences(x_test_sub, maxlen=max_len, padding='post', truncating='post')

word_index = imdb.get_word_index()
reverse_word_index = {v + 3: k for k, v in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(encoded_review):
    return ' '.join([reverse_word_index.get(i, '?') for i in encoded_review if i > 0])


model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128),
    
    SpatialDropout1D(0.25),
    
    Bidirectional(LSTM(64, dropout=0.25, recurrent_dropout=0.25)),
    
    Dense(32, activation='relu'),
    Dropout(0.4),
    
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
]

history = model.fit(
    x_train_padded, 
    y_train_sub,
    epochs=6,
    batch_size=64,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

test_loss, test_accuracy = model.evaluate(x_test_padded, y_test_sub, verbose=0)
print("\n" + "="*45)
print(f"Итоговая точность на тесте (Accuracy): {test_accuracy * 100:.2f}%")
print(f"Количество ошибок (Error Rate):        {(1 - test_accuracy) * 100:.2f}%")
print("="*45 + "\n")

y_pred_probs = model.predict(x_test_padded)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

print(classification_report(y_test_sub, y_pred, target_names=['Негативный', 'Позитивный']))

for i in range(5):
    prob = y_pred_probs[i][0]
    pred = "Позитив" if prob >= 0.5 else "Негатив"
    real = "Позитив" if y_test_sub[i] == 1 else "Негатив"
    print(f"Пример {i+1}:")
    print(f"Текст: {decode_review(x_test_sub[i])[:150]}...")
    print(f"Вероятность: {prob:.4f} | Предсказание: {pred} | Факт: {real} | Совпало: {'Да' if pred == real else 'Нет'}\n")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/6
125/125 ━━━━━━━━━━━━━━━━━━━━ 42s 320ms/step - accuracy: 0.5604 - loss: 0.6763 - val_accuracy: 0.7070 - val_loss: 0.5615
Epoch 2/6
125/125 ━━━━━━━━━━━━━━━━━━━━ 37s 297ms/step - accuracy: 0.7691 - loss: 0.5034 - val_accuracy: 0.8160 - val_loss: 0.4328
Epoch 3/6
125/125 ━━━━━━━━━━━━━━━━━━━━ 44s 351ms/step - accuracy: 0.8443 - loss: 0.3821 - val_accuracy: 0.8085 - val_loss: 0.4638
Epoch 4/6
125/125 ━━━━━━━━━━━━━━━━━━━━ 42s 337ms/step - accuracy: 0.8785 - loss: 0.3202 - val_accuracy: 0.8185 - val_loss: 0.5011

Итоговая точность на тесте (Accuracy): 80.52%
Количество ошибок (Error Rate):        19.48%

79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step
              precision    recall  f1-score   support

  Негативный       0.85      0.76      0.80      1308
  Позитивный       0.76      0.85      0.81      1192

    accuracy                           0.81      2500
   macro avg       0.81      0.81      0.81      2500
weighted avg       0.81      0.81      0.81      2500

Пример 1:
Текст: <S